In [2]:
import os

In [3]:
%pwd

'c:\\Users\\saqib\\OneDrive\\Desktop\\production-grade-wine-predictor\\research'

In [4]:
os.chdir("../")
%pwd

'c:\\Users\\saqib\\OneDrive\\Desktop\\production-grade-wine-predictor'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
!pip3 install PyYAML
!pip3 install joblib
%pip install ensure
#%pip install python-box


  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl (156 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached ensure-1.0.4-py3-none-any.whl.metadata (10 kB)
Using cached ensure-1.0.4-py3-none-any.whl (15 kB)
Note: you may need to restart the kernel to use updated packages.


In [9]:
import sys
!{sys.executable} -m pip install python-box
from box import ConfigBox
print("Success! ConfigBox is ready.")

  Using cached python_box-7.4.1-cp314-cp314-win_amd64.whl.metadata (8.3 kB)
Using cached python_box-7.4.1-cp314-cp314-win_amd64.whl (1.3 MB)
Success! ConfigBox is ready.


In [10]:
from src.productiongradewinepredictor.constants import *
from src.productiongradewinepredictor.utils.common import read_yaml, create_directories

In [11]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir

        )
        return data_ingestion_config

In [13]:
import os
import urllib.request as request
from src.productiongradewinepredictor import logger
import zipfile

In [14]:
## component-Data Ingestion

class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
    
    # Downloading the zip file
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [16]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-24 22:56:29,335: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-24 22:56:29,338: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-24 22:56:29,340: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-24 22:56:29,344: INFO: common: created directory at: artifacts]
[2026-03-24 22:56:29,346: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-24 22:56:35,671: INFO: 535005154: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: CF52:1499:1A3BF25:1C88D76:69C2D04D
Accept-Ranges: bytes
Date: Tue, 24 